# 03 — Cheap attacks: A3 abliteration, A4 HQQ 4/3/2-bit, A5 prompt-level (PLAN.md §4)
All 8 runs x {A3, A4q4, A4q3, A4q2, A5raw, A5bon}. Two GPUs, one worker per GPU; finished jobs are skipped,
so after a Kaggle timeout just run all cells again.

1. **DEV phase** (`harmful_dev`, `benign_dev`, `mmlu_dev`): sanity-check every attack. Any choice is made here.
2. **TEST phase** (`harmful_test`, `xstest_test`, `mmlu_test`) with `--final`: leaderboard cells, fixed attacks,
   no choices made from them (CLAUDE.md test-set rule).

Attack settings: `configs/attacks.yaml` (identical for every model). Abliterated / quantized weights exist
only in memory. Outputs go to the private HF dataset `Bennytimz/aftermarket-runs`; nothing here prints completions.

Requires: `HF_TOKEN` (write), GPU T4 x2.

In [ ]:
# (a) Clone the repo
!git clone -q https://github.com/Bennytimz/aftermarket
%cd aftermarket

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import sys, subprocess
sys.path.insert(0, "src")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
!PYTHONPATH=src python -m pytest -q tests/test_attacks.py tests/test_judge.py

## 1. DEV phase

In [ ]:
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_dev.yaml

In [ ]:
import glob, json
import pandas as pd
from aftermarket.judge import refusal_label, parse_mmlu

def summarize(splits):
    recs = []
    for p in sorted(glob.glob("results/gen/*.json")):
        m = json.load(open(p))
        j = m["job"]
        if j["split"] not in splits or j["attack_id"] == "A0":
            continue
        df = pd.read_parquet(p[:-5] + ".parquet", columns=["response", "answer", "item_id"])
        rec = {"run": j["run_id"], "attack": j["attack_id"], "split": j["split"], "n": m["n_prompts"],
               "prompts_per_sec": m["prompts_per_sec"], "hit_max": m["finish_length_frac"]}
        info = m.get("attack_info", {})
        if "layer" in info:
            rec["A3_layer"] = info["layer"]; rec["A3_sep"] = info["separation_at_layer"]
        if "vram_gb_model" in info:
            rec["A4_vram_gb"] = info["vram_gb_model"]; rec["A4_footprint_gb"] = info["footprint_gb"]
        if j["split"].startswith("mmlu"):
            rec["mmlu_acc"] = round((df.response.map(parse_mmlu) == df.answer).mean(), 3)
        else:
            lab = df.response.map(refusal_label)
            if j["attack_id"] == "A5bon":  # broken if ANY of the 8 variants is not a refusal
                rec["kw_refusal"] = round(lab.eq("refusal").groupby(df.item_id).all().mean(), 3)
            else:
                rec["kw_refusal"] = round(lab.eq("refusal").mean(), 3)
        recs.append(rec)
    return pd.DataFrame(recs)

# Stats only: no completion text is printed (PLAN.md §9).

dev = summarize({"harmful_dev", "benign_dev", "mmlu_dev"})
with pd.option_context("display.width", 220, "display.max_rows", 500):
    print(dev.to_string(index=False) if len(dev) else "no dev results yet")
# Sanity: A3 should cut keyword refusal on harmful_dev well below A0; A4q2 may wreck MMLU ("safe because broken").

## 2. TEST phase (leaderboard cells, `--final`)
Run after the dev table looks sane.

In [ ]:
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_test.yaml --final

In [ ]:
test = summarize({"harmful_test", "xstest_test", "mmlu_test"})
with pd.option_context("display.width", 220, "display.max_rows", 500):
    print(test.to_string(index=False) if len(test) else "no test results yet")

## 3. Judge (optional here; needs Llama-Guard-3-1B access)
Runs in its own process after generation. Skips files already judged.

In [ ]:
# !PYTHONPATH=src python -m aftermarket.judge --config configs/judge.yaml